# Klasifikasi DemogPairs Menggunakan ViT (Emosi dan Wajah) & Gaussian Naive Bayes

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
face_features = joblib.load('features/demogpairs_vit-face.pkl')
emotion_features = joblib.load('features/demogpairs_vit-emotion.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(face_features[key]) + list(emotion_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

100%|█████████████████████████████████████████████████████████████████████████| 10800/10800 [00:01<00:00, 10755.86it/s]

Jumlah fitur per gambar: 1536


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
var_smoothing_values = np.logspace(-9, 2, 40)  # dari 1e-9 sampai 1e2, 40 nilai

grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [GaussianNB()],
        'classifier__var_smoothing': var_smoothing_values
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',

}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

GaussianNB: 240 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models,
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix='models/clf_demogpairs_gnb_vit-emotion-face_',
    results_path='results/demogpairs_gnb_vit-emotion-face_'
)

sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: GaussianNB


{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.0058780160722749115), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}


Accuracy  : 0.8486111111111111
Precision : 0.8490074794512253
Recall    : 0.8486111111111111
F1 Score  : 0.848118117761148
               precision    recall  f1-score   support

Asian_Females     0.8842    0.8694    0.8768       360
  Asian_Males     0.8636    0.8972    0.8801       360
Black_Females     0.8338    0.7944    0.8137       360
  Black_Males     0.8557    0.9222    0.8877       360
White_Females     0.8659    0.7889    0.8256       360
  White_Males     0.7909    0.8194    0.8049       360

     accuracy                         0.8486      2160
    macro avg     0.8490    0.8486    0.8481      2160
 weighted avg     0.8490    0.8486    0.8481      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9592592592592593,0.884180790960452,0.8694444444444445,0.876750700280112,360
Asian_Males,0.9592592592592593,0.8636363636363636,0.8972222222222223,0.8801089918256131,360
Black_Females,0.9393518518518519,0.8338192419825073,0.7944444444444444,0.813655761024182,360
Black_Males,0.9611111111111111,0.8556701030927835,0.9222222222222223,0.8877005347593583,360
White_Females,0.9444444444444444,0.8658536585365854,0.7888888888888889,0.8255813953488372,360
White_Males,0.9337962962962963,0.7908847184986595,0.8194444444444444,0.8049113233287857,360


Confusion matrix saved: images\cm_gnb_vit-emotion-face_GaussianNB.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               313                 0                14                22                11                 0
         Asian_Males                 1               323                 3                 2                 9                22
       Black_Females                 5                 0               286                24                 5                40
         Black_Males                 6                 9                10               332                 1                 2
       White_Females                29                25                 8                 0               284                14
         White_Males                 0                17                22                 8                18               295


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
GaussianNB,models/clf_demogpairs_gnb_vit-emotion-face_GaussianNB.pkl,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.0058780160722749115), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8486111111111111,0.848118117761148,0.8490074794512253,0.8486111111111111,240


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_gnb_vit-emotion-face_GaussianNB.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 1784.0,
 'days': 0,
 'hours': 0,
 'minutes': 29,
 'seconds': 44.0,
 'text': '0 hari 0 jam 29 menit 44.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 10738.0,
 'days': 0,
 'hours': 2,
 'minutes': 58,
 'seconds': 58.0,
 'text': '0 hari 2 jam 58 menit 58.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.0058780160722749115), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8657,0.8576,0.8478,0.8507,0.8605,0.8565,0.8561,0.8572,0.8565,10.4011
2,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.011253355826007646), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8686,0.8519,0.8501,0.8513,0.8559,0.8556,0.8553,0.8568,0.8556,9.4657
3,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.021544346900318822), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8652,0.8466,0.8507,0.849,0.8536,0.853,0.853,0.8553,0.853,11.0794
4,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.00307029062975785), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8576,0.8559,0.8356,0.8466,0.8588,0.8509,0.8504,0.8517,0.8509,11.1449
...,...,...,...,...,...,...,...,...,...,...,...
237,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(100.0), 'pca': 'PCA', 'scaler': None}",0.8374,0.827,0.8252,0.8235,0.8229,0.8272,0.8251,0.8295,0.8272,7.3531
238,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(27.283333764867695), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8339,0.8264,0.8194,0.8275,0.827,0.8269,0.8256,0.8275,0.8269,20.2997
239,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(52.233450742668325), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8339,0.8258,0.8189,0.8275,0.827,0.8266,0.8254,0.8274,0.8266,12.0994
240,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(100.0), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8333,0.8247,0.8189,0.8275,0.8264,0.8262,0.8249,0.8269,0.8262,14.371
